# Backbone Latent-Space Analysis

Load final checkpoints from the registered experiment protocol, extract held-out GTSRB representations, learn two-dimensional autoencoder projections, and compare class clustering and expert routing. Model construction, data selection, transforms, class mappings, and checkpoint naming follow `init_experiment.py`; this notebook does not redefine experiment components.

In [1]:
import os
import subprocess
import sys
from dataclasses import replace
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    drive = None
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/JHU_IS_26')
    if not PROJECT_DIR.exists():
        subprocess.run(['git', 'clone', 'https://github.com/ddimpfel/JHU_IS_26.git', str(PROJECT_DIR)], check=True)
else:
    current = Path.cwd().resolve()
    candidates = (current, current / 'Current Work', current.parent, current.parent / 'Current Work')
    PROJECT_DIR = next((path for path in candidates if (path / 'init_experiment.py').exists()), None)
    if PROJECT_DIR is None:
        raise FileNotFoundError('Run from Current Work, its notebooks directory, or the workspace root.')

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
print(f'Project directory: {PROJECT_DIR}')

Project directory: C:\1SCHOOL\grad\12-IS_S26\Current Work


In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from IPython.display import display
from sklearn.manifold import trustworthiness
from sklearn.metrics import davies_bouldin_score, silhouette_score
from sklearn.preprocessing import StandardScaler
from torch.optim import AdamW
from torch.utils.data import DataLoader, TensorDataset

from continual_learning import CILComputerVisionModel
from init_experiment import (
    ExperimentConfig,
    JointEmbeddingRouterExperts,
    build_gtsrb_data,
    build_model_from_name,
    load_sign_names,
    set_reproducibility,
    strip_checkpoint_suffix,
)

sns.set_theme(style='whitegrid')

c:\1SCHOOL\grad\12-IS_S26\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Analysis configuration

Checkpoint filenames use the final-run format produced by the experiment runner: `<registered model name>__seed-<seed>.pth`.

In [3]:
SEED = 7
MODELS_DIR = PROJECT_DIR / 'models'
JE_FEATURE_SOURCE = 'projections'  # projections, embeddings, or backbone_features
FEATURE_SAMPLE_LIMIT = None  # e.g. 4000 for a quicker exploratory run
AUTOENCODER_HIDDEN_DIM = 256
AUTOENCODER_LATENT_DIM = 2
AUTOENCODER_EPOCHS = 80
AUTOENCODER_BATCH_SIZE = 256 if IN_COLAB else 64
AUTOENCODER_LR = 1e-3
AUTOENCODER_WEIGHT_DECAY = 1e-5

SELECTED_MODEL_FILES = [
    'ConvNeXt Tiny + MLP Router + MLP Experts__seed-7.pth',
    'JE ConvNeXt Tiny + MLP Router + MLP Experts__seed-7.pth',
    'ConvNeXt Tiny + Regression Router + Transformer Experts__seed-7.pth',
    'JE ConvNeXt Tiny + Regression Router + Transformer Experts__seed-7.pth',
    'MobileNet Large + MLP Router + MLP Experts__seed-7.pth',
    'JE MobileNet Large + MLP Router + MLP Experts__seed-7.pth',
    'MobileNet Large + Regression Router + Transformer Experts__seed-7.pth',
    'JE MobileNet Large + Regression Router + Transformer Experts__seed-7.pth',
]

config = replace(ExperimentConfig(seed=SEED), batch_size=64 if IN_COLAB else 16)
device = config.resolved_device
np_rng, torch_generator = set_reproducibility(SEED)
print(f'Device: {device}')
print(f'Model directory: {MODELS_DIR.resolve()}')

Device: cpu
Model directory: C:\1SCHOOL\grad\12-IS_S26\Current Work\models


## Registered held-out data and checkpoints

In [4]:
data = build_gtsrb_data(config)
analysis_dataloader = data.test_loader()
sign_names = load_sign_names(PROJECT_DIR / 'signnames.csv')
class_name_lookup = {data.class_mapping[row.ClassId]: row.SignName for row in sign_names.itertuples() if row.ClassId in data.class_mapping}

selected_checkpoint_paths = [MODELS_DIR / filename for filename in SELECTED_MODEL_FILES]
missing_checkpoints = [path.name for path in selected_checkpoint_paths if not path.exists()]
if missing_checkpoints:
    raise FileNotFoundError(
        f'Add the selected final checkpoints to {MODELS_DIR.resolve()}. Missing: {missing_checkpoints}'
    )
print(f'Held-out examples: {len(data.test_dataset)}')
print(f'Selected checkpoints: {len(selected_checkpoint_paths)}')

FileNotFoundError: Add the selected final checkpoints to C:\1SCHOOL\grad\12-IS_S26\Current Work\models. Missing: ['ConvNeXt Tiny + MLP Router + MLP Experts__seed-7.pth', 'JE ConvNeXt Tiny + MLP Router + MLP Experts__seed-7.pth', 'ConvNeXt Tiny + Regression Router + Transformer Experts__seed-7.pth', 'JE ConvNeXt Tiny + Regression Router + Transformer Experts__seed-7.pth', 'MobileNet Large + MLP Router + MLP Experts__seed-7.pth', 'JE MobileNet Large + MLP Router + MLP Experts__seed-7.pth', 'MobileNet Large + Regression Router + Transformer Experts__seed-7.pth', 'JE MobileNet Large + Regression Router + Transformer Experts__seed-7.pth']

## Feature extraction

In [ ]:
def extract_features(model, images):
    if isinstance(model, JointEmbeddingRouterExperts):
        encoded = model.generalist.backbone.encode(images)
        if JE_FEATURE_SOURCE not in encoded:
            raise KeyError(f'Unknown JE feature source {JE_FEATURE_SOURCE!r}; available: {list(encoded)}')
        features = encoded[JE_FEATURE_SOURCE]
        source = f'JE {JE_FEATURE_SOURCE}'
    else:
        _, features = model.generalist(images, return_features=True)
        source = 'Backbone Features'
    return torch.flatten(features, 1), source

def load_checkpoint(path):
    model_name = strip_checkpoint_suffix(path.stem)
    model = build_model_from_name(model_name, config)
    wrapper = CILComputerVisionModel.load(model, filename=str(path), map_location=device, device=device)
    wrapper.model.eval()
    return model_name, wrapper

def collect_feature_bundles(checkpoint_paths):
    rows, bundles = [], {}
    for path in checkpoint_paths:
        model_name, wrapper = load_checkpoint(path)
        features_list, labels_list, paths = [], [], []
        feature_source = None
        with torch.no_grad():
            for images, targets, batch_paths in analysis_dataloader:
                images = torch.stack(images).to(device)
                batch_features, feature_source = extract_features(wrapper.model, images)
                features_list.append(batch_features.cpu())
                labels_list.append(torch.tensor([int(target['label']) for target in targets]))
                paths.extend(batch_paths)
        features = torch.cat(features_list).numpy()
        labels = torch.cat(labels_list).numpy()
        if FEATURE_SAMPLE_LIMIT is not None and len(features) > FEATURE_SAMPLE_LIMIT:
            indices = np_rng.choice(len(features), FEATURE_SAMPLE_LIMIT, replace=False)
            features, labels = features[indices], labels[indices]
            paths = [paths[index] for index in indices]
        key = path.stem
        backbone = model_name.split(' + ')[0]
        bundles[key] = {'model_name': model_name, 'checkpoint': path, 'features': features, 'labels': labels, 'paths': paths, 'feature_source': feature_source, 'backbone': backbone}
        rows.append({'Run': key, 'Model': model_name, 'Backbone': backbone, 'Feature Source': feature_source, 'Feature Dimension': features.shape[1], 'Sample Count': len(features)})
        del wrapper
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return pd.DataFrame(rows), bundles

feature_summary_df, feature_bundles = collect_feature_bundles(selected_checkpoint_paths)
display(feature_summary_df)

## Autoencoder projections and cluster metrics

In [ ]:
class ProjectionAutoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, latent_dim=2):
        super().__init__()
        inner_dim = min(hidden_dim, max(64, input_dim // 2))
        self.encoder = nn.Sequential(nn.Linear(input_dim, inner_dim), nn.GELU(), nn.Linear(inner_dim, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, latent_dim))
        self.decoder = nn.Sequential(nn.Linear(latent_dim, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, inner_dim), nn.GELU(), nn.Linear(inner_dim, input_dim))
    def forward(self, values):
        latent = self.encoder(values)
        return self.decoder(latent), latent

def fit_autoencoder(features):
    standardized = StandardScaler().fit_transform(features)
    tensor = torch.tensor(standardized, dtype=torch.float32)
    loader = DataLoader(TensorDataset(tensor), batch_size=AUTOENCODER_BATCH_SIZE, shuffle=True, generator=torch_generator)
    model = ProjectionAutoencoder(tensor.shape[1], AUTOENCODER_HIDDEN_DIM, AUTOENCODER_LATENT_DIM).to(device)
    optimizer = AdamW(model.parameters(), lr=AUTOENCODER_LR, weight_decay=AUTOENCODER_WEIGHT_DECAY)
    losses = []
    for _ in range(AUTOENCODER_EPOCHS):
        total = 0.0
        model.train()
        for (batch,) in loader:
            batch = batch.to(device)
            optimizer.zero_grad(set_to_none=True)
            reconstruction, _ = model(batch)
            loss = nn.functional.mse_loss(reconstruction, batch)
            loss.backward(); optimizer.step()
            total += loss.item() * len(batch)
        losses.append(total / len(tensor))
    model.eval()
    with torch.no_grad():
        _, latent = model(tensor.to(device))
    return {'autoencoder': model, 'loss_history': losses, 'latent': latent.cpu().numpy(), 'standardized_features': standardized}

projection_results = {}
for run, bundle in feature_bundles.items():
    print(f'Training projection for {run}...')
    projection_results[run] = {**bundle, **fit_autoencoder(bundle['features'])}

In [ ]:
metric_rows = []
for run, result in projection_results.items():
    latent, labels = result['latent'], result['labels']
    valid = len(np.unique(labels)) > 1 and len(latent) > len(np.unique(labels))
    metric_rows.append({
        'Run': run, 'Model': result['model_name'], 'Backbone': result['backbone'], 'Feature Source': result['feature_source'],
        'Final Reconstruction Loss': result['loss_history'][-1],
        'Silhouette Score': float(silhouette_score(latent, labels)) if valid else np.nan,
        'Davies-Bouldin Score': float(davies_bouldin_score(latent, labels)) if valid else np.nan,
        'Trustworthiness': float(trustworthiness(result['standardized_features'], latent, n_neighbors=min(10, len(latent) - 1))) if len(latent) > 1 else np.nan,
    })
cluster_metrics_df = pd.DataFrame(metric_rows).sort_values(['Silhouette Score', 'Trustworthiness'], ascending=False).reset_index(drop=True)
display(cluster_metrics_df)

## Latent-space visualizations

In [ ]:
for run, result in projection_results.items():
    frame = pd.DataFrame({'Latent X': result['latent'][:, 0], 'Latent Y': result['latent'][:, 1], 'Class ID': result['labels']})
    frame['Class'] = frame['Class ID'].map(class_name_lookup)
    plt.figure(figsize=(10, 7))
    sns.scatterplot(data=frame, x='Latent X', y='Latent Y', hue='Class', palette='tab10', s=18, alpha=0.7)
    plt.title(f"{run} — {result['feature_source']}")
    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

plt.figure(figsize=(10, max(4, len(cluster_metrics_df) * 0.55)))
sns.barplot(data=cluster_metrics_df, y='Run', x='Silhouette Score', hue='Backbone', dodge=False)
plt.title('Latent Class Separation by Final Checkpoint')
plt.tight_layout()
plt.show()

## Primary-expert routing overlays

In [ ]:
def collect_routing(result):
    _, wrapper = load_checkpoint(result['checkpoint'])
    routing_rows = []
    with torch.no_grad():
        for images, targets, paths in analysis_dataloader:
            wrapper.model.reset_routing_state()
            _ = wrapper.model(torch.stack(images).to(device))
            topk = wrapper.model._last_topk_indices.cpu().numpy()
            probabilities = wrapper.model._last_router_probs.cpu().numpy()
            for index, path in enumerate(paths):
                expert = int(topk[index, 0])
                routing_rows.append({'Path': path, 'Primary Expert': f'E{expert}', 'Primary Expert Probability': float(probabilities[index, expert])})
    routing = pd.DataFrame(routing_rows).set_index('Path').reindex(result['paths'])
    if routing.isna().any().any():
        raise ValueError(f"Routing rows could not be aligned for {result['checkpoint'].name}")
    return routing.reset_index()

for run, result in projection_results.items():
    routing = collect_routing(result)
    routing['Latent X'] = result['latent'][:, 0]
    routing['Latent Y'] = result['latent'][:, 1]
    plt.figure(figsize=(9, 7))
    sns.scatterplot(data=routing, x='Latent X', y='Latent Y', hue='Primary Expert', palette='tab10', s=18, alpha=0.7)
    plt.title(f'{run} — Primary Expert')
    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.show()